<a href="https://colab.research.google.com/github/lovnishverma/Python-Getting-Started/blob/main/RNN_text_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple RNN for Text Classification
M.TECH AI - Practical 5

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

## 1. Dataset

In [2]:
texts = ["i love this", "i hate this"]
labels = [1, 0]

## 2. Text to Numbers

In [3]:
vocab = {"i":0, "love":1, "hate":2, "this":3}

def encode(text):
    return [vocab[word] for word in text.split()]

X = [encode(t) for t in texts]
y = torch.tensor(labels, dtype=torch.float32)

## 3. Padding

In [4]:
max_len = max(len(seq) for seq in X)

def pad(seq):
    return seq + [0]*(max_len - len(seq))

X = torch.tensor([pad(seq) for seq in X])

## 4. Model

In [5]:
class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(4, 8)
        self.rnn = nn.RNN(8, 16, batch_first=True)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return torch.sigmoid(out).squeeze()

model = SimpleRNN()

## 5. Training

In [6]:
loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(50):
    preds = model(X)
    loss = loss_fn(preds, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 0.5526
Epoch 20, Loss: 0.1626
Epoch 30, Loss: 0.0277
Epoch 40, Loss: 0.0096
Epoch 50, Loss: 0.0053


## 6. Testing

In [7]:
test = torch.tensor([pad(encode("i love this"))])
print("Prediction:", model(test).item())

Prediction: 0.9947726130485535
